In [5]:
import pandas as pd
import numpy as np
import FinanceDataReader as fdr
from datetime import datetime, timedelta
import pymysql
from tqdm.auto import tqdm
from DATA.stock_invest_function import get_db_host
import requests

# ============================================================
# A. 공통 유틸 함수들
# ============================================================

def calculate_returns_and_volatility(df, price_column='Close'):
    """
    일별 로그수익률 + 20/60/120/252일 수익률 & 변동성(연율화) 계산
    """
    result_df = df.copy()
    result_df['Daily_Return'] = np.log(result_df[price_column] / result_df[price_column].shift(1))

    periods = {20: '20D', 60: '60D', 120: '120D', 252: '252D'}

    for days, label in periods.items():
        # 기간 수익률
        result_df[f'Return_{label}'] = result_df[price_column].pct_change(periods=days)
        # 기간 변동성(연율화)
        result_df[f'Volatility_{label}'] = (
            result_df['Daily_Return'].rolling(window=days).std() * np.sqrt(252)
        )

    return result_df


def calculate_beta(stock_df, market_df, periods=[252, 756, 1260]):
    """
    1년/3년/5년 rolling 베타 계산
    stock_df, market_df 는 반드시 같은 날짜 index 로 맞춰서 들어오는 게 이상적임
    """
    result_df = stock_df.copy()
    period_names = {252: '1Y', 756: '3Y', 1260: '5Y'}

    for period in periods:
        period_name = period_names.get(period, f'{period}D')
        beta_values = []

        for i in range(len(result_df)):
            if i < period - 1:
                beta_values.append(np.nan)
            else:
                stock_returns = result_df['Daily_Return'].iloc[i-period+1:i+1].values
                market_returns = market_df['Daily_Return'].iloc[i-period+1:i+1].values

                mask = ~(np.isnan(stock_returns) | np.isnan(market_returns))

                if mask.sum() < 20:
                    beta_values.append(np.nan)
                else:
                    stock_returns_clean = stock_returns[mask]
                    market_returns_clean = market_returns[mask]

                    covariance = np.cov(stock_returns_clean, market_returns_clean)[0, 1]
                    market_variance = np.var(market_returns_clean)

                    if market_variance == 0:
                        beta_values.append(np.nan)
                    else:
                        beta = covariance / market_variance
                        beta_values.append(beta)

        result_df[f'Beta_{period_name}'] = beta_values

    return result_df


def get_korea_5y_treasury_rate(start_date, end_date, api_key):
    """
    한국은행 ECOS API에서 5년 국고채(일별) 금리 가져오기
    반환: pandas.Series (index=날짜, 값=금리)
    """
    start_date_str = start_date.replace('-', '')
    end_date_str = end_date.replace('-', '')

    if api_key is None:
        print("\n⚠️ API 키 없음 - 대용치(3.2%) 사용")
        return None

    print(f"\n[한국은행 API] 5년 국채금리 수집: {start_date} ~ {end_date}")

    try:
        stat_code = "817Y002"   # 국고채(일별)
        item_code = "010210000"  # 5년
        url = (
            f"https://ecos.bok.or.kr/api/StatisticSearch/"
            f"{api_key}/json/kr/1/100000/{stat_code}/D/"
            f"{start_date_str}/{end_date_str}/{item_code}"
        )

        response = requests.get(url, timeout=10)

        if response.status_code == 200:
            data = response.json()

            if 'StatisticSearch' in data and 'row' in data['StatisticSearch']:
                rows = data['StatisticSearch']['row']
                df = pd.DataFrame(rows)
                df['Date'] = pd.to_datetime(df['TIME'])
                df['Rate'] = pd.to_numeric(df['DATA_VALUE'])
                df = df.set_index('Date')[['Rate']]

                print(f"✓ 수집 완료 - {len(df)}일, 평균 {df['Rate'].mean():.3f}%")
                return df['Rate']

        print("✗ 데이터 수집 실패")
        return None

    except Exception as e:
        print(f"✗ 에러: {e}")
        return None


def calculate_required_return(stock_df, market_df, api_key=None):
    """
    주주 요구수익률 계산 (CAPM)
    - Risk_Free_Rate    : 5년 국채(한국은행)
    - Market_Return     : KOSPI 5년 연평균 수익률
    - Market_Risk_Premium = Rm - Rf
    - Required_Return   : Rf + (Rm - Rf) * Beta_5Y
    """
    print("\n" + "=" * 80)
    print("주주 요구수익률 계산")
    print("=" * 80)

    result_df = stock_df.copy()

    # ------------------------
    # 1) 무위험이자율 (5년 국채)
    # ------------------------
    print("\n[1단계] 무위험이자율")
    start_date = result_df.index[0].strftime('%Y-%m-%d')
    end_date   = result_df.index[-1].strftime('%Y-%m-%d')

    treasury_5y = get_korea_5y_treasury_rate(start_date, end_date, api_key)

    if treasury_5y is not None and not treasury_5y.empty:
        treasury_df    = pd.DataFrame({'Treasury_5Y': treasury_5y})
        stock_dates_df = pd.DataFrame(index=result_df.index)
        combined       = pd.concat([stock_dates_df, treasury_df], axis=1)
        combined['Treasury_5Y'] = combined['Treasury_5Y'].ffill().bfill()
        risk_free_rate = combined['Treasury_5Y']
        print("✓ 한국은행 API 데이터 연동 완료")
    else:
        risk_free_rate = pd.Series(3.2, index=result_df.index)
        print("  → API 실패 또는 데이터 없음 → 대용치 3.2% 사용")

    result_df['Risk_Free_Rate'] = risk_free_rate
    print(
        f"  평균: {risk_free_rate.mean():.3f}%, "
        f"범위: {risk_free_rate.min():.3f}% ~ {risk_free_rate.max():.3f}%"
    )

    # ------------------------
    # 2) 시장수익률 (5년 연평균, KOSPI 기준)
    # ------------------------
    print("\n[2단계] 시장수익률 (5년)")
    period = 5
    days   = period * 252

    # stock_df 의 날짜 index 기준으로 KOSPI 가격 정렬
    aligned_market = market_df.reindex(result_df.index).ffill().bfill()

    annual_returns = []
    for i in range(len(result_df)):
        if i < days - 1:
            annual_returns.append(np.nan)
        else:
            start_price = aligned_market['Close'].iloc[i - days + 1]
            end_price   = aligned_market['Close'].iloc[i]

            if start_price > 0:
                annualized_return = (end_price / start_price) ** (1 / period) - 1
                annual_returns.append(annualized_return * 100)
            else:
                annual_returns.append(np.nan)

    result_df['Market_Return'] = annual_returns
    print(f"✓ 시장수익률 평균: {np.nanmean(annual_returns):.3f}%")

    # ------------------------
    # 3) 베타
    # ------------------------
    print("\n[3단계] 베타 확인")
    if 'Beta_5Y' not in result_df.columns:
        print("✗ Beta_5Y 없음 → 요구수익률 계산 불가 (그대로 반환)")
        return result_df

    print(f"✓ Beta_5Y 평균: {result_df['Beta_5Y'].mean():.3f}")

    # ------------------------
    # 4) 요구수익률 계산 (Required_Return)
    # ------------------------
    print("\n[4단계] 요구수익률 계산")
    result_df['Market_Risk_Premium'] = (
        result_df['Market_Return'] - result_df['Risk_Free_Rate']
    )

    result_df['Required_Return'] = (
        result_df['Risk_Free_Rate'] +
        result_df['Market_Risk_Premium'] * result_df['Beta_5Y']
    )

    # COE < Rf 인 구간 보정
    below_rf_mask  = result_df['Required_Return'] < result_df['Risk_Free_Rate']
    below_rf_count = below_rf_mask.sum()

    if below_rf_count > 0:
        print(f"\n⚠️ 요구수익률 < Rf 인 관측치: {below_rf_count}개 → Rf로 대체")
        result_df.loc[below_rf_mask, 'Required_Return'] = \
            result_df.loc[below_rf_mask, 'Risk_Free_Rate']

    rr_mean = result_df['Required_Return'].mean()
    rr_min  = result_df['Required_Return'].min()
    rr_max  = result_df['Required_Return'].max()

    print(f"✓ 완료 - 평균: {rr_mean:.3f}%, 범위: {rr_min:.3f}% ~ {rr_max:.3f}%")
    print("=" * 80)

    return result_df


# ============================================================
# B. DB 저장 및 조회 함수
# ============================================================

def insert_required_return_to_db(long_df: pd.DataFrame,
                                 db_info: dict,
                                 table_name: str = "korea_required_return_result",
                                 chunk_size: int = 50_000):
    """
    long_df(date, ticker, indicator, value)를 mariaDB에 upsert 저장
    """
    total_rows = len(long_df)
    if total_rows == 0:
        print("※ 저장할 데이터가 없습니다.")
        return

    # ---- 디버그: 저장 전 샘플 ----
    print("\n[DEBUG] 저장할 데이터 샘플:")
    print(long_df.head(10))
    print("\n[DEBUG] dtypes:")
    print(long_df.dtypes)

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4",
        autocommit=False
    )

    try:
        with conn.cursor() as cur:
            create_sql = f"""
            CREATE TABLE IF NOT EXISTS {table_name} (
                date DATE NOT NULL,
                ticker VARCHAR(20) NOT NULL,
                indicator VARCHAR(50) NOT NULL,
                value DOUBLE,
                PRIMARY KEY (date, ticker, indicator)
            ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
            """
            cur.execute(create_sql)

            insert_sql = f"""
            INSERT INTO {table_name} (date, ticker, indicator, value)
            VALUES (%s, %s, %s, %s)
            ON DUPLICATE KEY UPDATE
                value = VALUES(value);
            """

            for start in range(0, total_rows, chunk_size):
                end   = min(start + chunk_size, total_rows)
                chunk = long_df.iloc[start:end]

                data = []
                for _, row in chunk.iterrows():
                    d = row["date"]
                    if isinstance(d, pd.Timestamp):
                        d = d.strftime("%Y-%m-%d")

                    data.append((
                        d,
                        str(row["ticker"]),
                        str(row["indicator"]),
                        float(row["value"]) if pd.notnull(row["value"]) else None
                    ))

                # 첫 chunk 샘플 출력
                if start == 0:
                    print("\n[DEBUG] 실제 INSERT될 첫 5개:")
                    for i, item in enumerate(data[:5]):
                        print(f"  Row {i}: {item}")

                cur.executemany(insert_sql, data)
                conn.commit()
                print(f"   - DB chunk 저장: rows {start}~{end-1} (총 {end-start}행)")

        print(f"\n✓ DB 저장 완료: {total_rows} rows → {table_name}")

    except Exception as e:
        conn.rollback()
        print("\n✗ DB 저장 중 오류:", e)
        import traceback
        traceback.print_exc()

    finally:
        conn.close()


def test_db_query(db_info: dict,
                  table_name: str = "korea_required_return_result",
                  ticker: str = "005930"):
    """
    DB에서 최근 20행을 조회해서 저장이 정상인지 확인
    """
    print("\n" + "=" * 80)
    print(f"DB 조회 테스트: ticker={ticker}")
    print("=" * 80)

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        query = f"""
        SELECT date, ticker, indicator, value
        FROM {table_name}
        WHERE ticker = %s
        ORDER BY date DESC, indicator
        LIMIT 20
        """

        df = pd.read_sql(query, conn, params=(ticker,))

        print("\n[조회 결과 - 최근 20행]")
        print(df)

        print("\n[데이터 타입]")
        print(df.dtypes)

        print("\n[유니크 indicator]")
        print(df['indicator'].unique())

    except Exception as e:
        print("\n✗ 조회 오류:", e)
        import traceback
        traceback.print_exc()

    finally:
        conn.close()


# ============================================================
# C. 배치 실행 함수 (test_mode / full 모드)
# ============================================================

def run_required_return_batch(batch_size=10, test_mode=True, tickers_override=None):
    """
    test_mode=True  : KRX 상장사 중 앞 10개만 테스트용으로 처리 + DB조회
    test_mode=False : 전체 상장사 처리
    """
    bok_key          = "O6FJIBYZCWZF9ZFHL6F9"
    end_date         = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')
    start_date       = "2014-01-01"
    min_required_start = pd.Timestamp("2019-01-01")  # 이 이후 상장 종목은 제외

    db_info = {
        'host': get_db_host(),
        'port': 3307,
        'user': 'stox7412',
        'password': 'Apt106503!~',
        'database': 'investar'
    }

    # 1) KOSPI 데이터
    print("\n" + "=" * 80)
    print("KOSPI 지수 데이터 수집 및 분석")
    print("=" * 80)

    market_raw = fdr.DataReader('KS11', start_date, end_date)
    if market_raw is None or len(market_raw) == 0:
        print("✗ KOSPI 데이터 수집 실패")
        return

    market_analyzed = calculate_returns_and_volatility(market_raw)
    print(f"✓ KOSPI 데이터 수집 완료: {len(market_analyzed)} rows")

    # 2) KRX 상장사 리스트
    print("\n" + "=" * 80)
    print("KRX 상장사 리스트 수집")
    print("=" * 80)

    krx_list = fdr.StockListing('KRX')
    tickers  = krx_list["Code"].dropna().tolist()

    # ---------------------------
    # ① 사용자가 tickers_override 를 주면 그것만 사용
    # ---------------------------
    if tickers_override is not None:
        tickers = tickers_override
        print(f"⚠️ 사용자 지정 tickers {len(tickers)}개만 처리합니다:")
        print(tickers)
    # ---------------------------
    # ② test_mode=True 일 때는 자동 10개
    # ---------------------------
    elif test_mode:
        tickers = tickers[:10]
        print(f"⚠️ 테스트 모드: {len(tickers)}개 종목만 자동 처리")

    print(f"예시 tickers: {tickers[:10]}")

    batch_list  = []
    batch_index = 0
    total_rows  = 0
    test_ticker = None  # 테스트용 조회 ticker

    for i, code in enumerate(tqdm(tickers, desc="Processing tickers"), start=1):
        print("\n" + "-" * 80)
        print(f"[{code}] 처리 시작")
        print("-" * 80)

        try:
            stock_raw = fdr.DataReader(code, start_date, end_date)
            if stock_raw is None or len(stock_raw) == 0:
                print("  → 주가 데이터 없음 → 스킵")
                continue

            if stock_raw.index.min() > min_required_start:
                print("  → 상장일이 너무 최근 → 스킵")
                continue

            # 개별 종목 분석
            stock_analyzed   = calculate_returns_and_volatility(stock_raw)

            # index 기준으로 KOSPI와 align (길이 다를 수 있으므로)
            aligned_market   = market_analyzed.reindex(stock_analyzed.index).ffill().bfill()

            stock_with_beta  = calculate_beta(stock_analyzed, aligned_market)
            stock_rr         = calculate_required_return(stock_with_beta,
                                                         aligned_market,
                                                         api_key=bok_key)

            metric_cols = [
                'Risk_Free_Rate', 'Market_Return', 'Market_Risk_Premium', 'Required_Return',
                'Beta_1Y', 'Beta_3Y', 'Beta_5Y',
                'Return_20D', 'Return_60D', 'Return_120D', 'Return_252D',
                'Volatility_20D', 'Volatility_60D', 'Volatility_120D', 'Volatility_252D',
            ]
            metric_cols = [c for c in metric_cols if c in stock_rr.columns]

            if not metric_cols:
                print("  → 저장할 metric 없음 → 스킵")
                continue

            print(f"  → 저장할 컬럼: {metric_cols}")

            wide = stock_rr[metric_cols].reset_index()
            wide = wide.rename(columns={wide.columns[0]: "date"})

            long_df = wide.melt(
                id_vars=['date'],
                value_vars=metric_cols,
                var_name='indicator',
                value_name='value'
            )
            long_df['ticker'] = code
            long_df = long_df[['date', 'ticker', 'indicator', 'value']]

            batch_list.append(long_df)
            total_rows += len(long_df)

            if test_ticker is None:
                test_ticker = code  # 첫 성공 종목 저장

            print(f"  ✓ {code} 변환 완료: {len(long_df)} rows")

            # --- 배치 단위 저장 ---
            if (i % batch_size == 0) and batch_list:
                batch_index += 1
                merged = pd.concat(batch_list, ignore_index=True)
                print(f"\n[Batch Save] Batch {batch_index} - rows={len(merged)}")
                insert_required_return_to_db(merged, db_info)
                batch_list = []

        except Exception as e:
            print("  ✗ 오류 발생:", e)
            import traceback
            traceback.print_exc()
            continue

    # --- 마지막 남은 배치 저장 ---
    if batch_list:
        batch_index += 1
        merged = pd.concat(batch_list, ignore_index=True)
        print(f"\n[Batch Save] FINAL Batch {batch_index} - rows={len(merged)}")
        insert_required_return_to_db(merged, db_info)

    print("\n" + "=" * 80)
    print(f"전체 저장 완료! 총 저장 row 수: {total_rows}")
    print("=" * 80)

    # 테스트 모드일 때는 DB 조회까지 자동 수행
    if test_mode and test_ticker:
        test_db_query(db_info, ticker=test_ticker)


# ============================================================
# D. 실행 예시
# ============================================================

if __name__ == "__main__":
    # 1) 테스트 모드: 앞 10개 종목만 처리 + DB 조회 테스트
    # run_required_return_batch(batch_size=5, test_mode=True)

    # 2) 전체 실행 (테스트 모드 확인 후 주석 해제해서 사용)
    run_required_return_batch(batch_size=50, test_mode=False)



KOSPI 지수 데이터 수집 및 분석
✓ KOSPI 데이터 수집 완료: 2919 rows

KRX 상장사 리스트 수집
예시 tickers: ['005930', '000660', '373220', '207940', '005935', '005380', '329180', '034020', '105560', '012450']


Processing tickers:   0%|          | 0/2886 [00:00<?, ?it/s]


--------------------------------------------------------------------------------
[005930] 처리 시작
--------------------------------------------------------------------------------

주주 요구수익률 계산

[1단계] 무위험이자율

[한국은행 API] 5년 국채금리 수집: 2014-01-02 ~ 2025-11-21
✓ 수집 완료 - 2930일, 평균 2.524%
✓ 한국은행 API 데이터 연동 완료
  평균: 2.524%, 범위: 1.172% ~ 4.632%

[2단계] 시장수익률 (5년)
✓ 시장수익률 평균: 3.942%

[3단계] 베타 확인
✓ Beta_5Y 평균: 1.133

[4단계] 요구수익률 계산

⚠️ 요구수익률 < Rf 인 관측치: 632개 → Rf로 대체
✓ 완료 - 평균: 5.165%, 범위: 1.172% ~ 13.311%
  → 저장할 컬럼: ['Risk_Free_Rate', 'Market_Return', 'Market_Risk_Premium', 'Required_Return', 'Beta_1Y', 'Beta_3Y', 'Beta_5Y', 'Return_20D', 'Return_60D', 'Return_120D', 'Return_252D', 'Volatility_20D', 'Volatility_60D', 'Volatility_120D', 'Volatility_252D']
  ✓ 005930 변환 완료: 43785 rows

--------------------------------------------------------------------------------
[000660] 처리 시작
--------------------------------------------------------------------------------

주주 요구수익률 계산

[1단계] 무위험이자율

[한국은행 API] 5년

Traceback (most recent call last):
  File "C:\Users\82108\AppData\Local\Temp\ipykernel_26108\2131055907.py", line 439, in run_required_return_batch
    stock_raw = fdr.DataReader(code, start_date, end_date)
  File "C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\FinanceDataReader\data.py", line 92, in DataReader
    return YahooDailyReader(codes, start, end).read()
  File "C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\FinanceDataReader\yahoo\data.py", line 61, in read
    return _yahoo_data_reader(self.symbol, self.exchange, self.start, self.end)
  File "C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\FinanceDataReader\yahoo\data.py", line 38, in _yahoo_data_reader
    r.raise_for_status()
  File "C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\requests\models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 404 Client Er


주주 요구수익률 계산

[1단계] 무위험이자율

[한국은행 API] 5년 국채금리 수집: 2014-01-02 ~ 2025-11-21
✓ 수집 완료 - 2930일, 평균 2.524%
✓ 한국은행 API 데이터 연동 완료
  평균: 2.524%, 범위: 1.172% ~ 4.632%

[2단계] 시장수익률 (5년)
✓ 시장수익률 평균: 3.942%

[3단계] 베타 확인
✓ Beta_5Y 평균: 1.418

[4단계] 요구수익률 계산

⚠️ 요구수익률 < Rf 인 관측치: 632개 → Rf로 대체
✓ 완료 - 평균: 5.789%, 범위: 1.172% ~ 15.721%
  → 저장할 컬럼: ['Risk_Free_Rate', 'Market_Return', 'Market_Risk_Premium', 'Required_Return', 'Beta_1Y', 'Beta_3Y', 'Beta_5Y', 'Return_20D', 'Return_60D', 'Return_120D', 'Return_252D', 'Volatility_20D', 'Volatility_60D', 'Volatility_120D', 'Volatility_252D']
  ✓ 086520 변환 완료: 43785 rows

--------------------------------------------------------------------------------
[003230] 처리 시작
--------------------------------------------------------------------------------

주주 요구수익률 계산

[1단계] 무위험이자율

[한국은행 API] 5년 국채금리 수집: 2014-01-02 ~ 2025-11-21
✓ 수집 완료 - 2930일, 평균 2.524%
✓ 한국은행 API 데이터 연동 완료
  평균: 2.524%, 범위: 1.172% ~ 4.632%

[2단계] 시장수익률 (5년)
✓ 시장수익률 평균: 3.942%

[3단계] 베타 확인
✓ Beta_5Y 평균

...


주주 요구수익률 계산

[1단계] 무위험이자율

[한국은행 API] 5년 국채금리 수집: 2018-07-13 ~ 2025-11-21
✓ 수집 완료 - 1814일, 평균 2.591%
✓ 한국은행 API 데이터 연동 완료
  평균: 2.591%, 범위: 1.172% ~ 4.632%

[2단계] 시장수익률 (5년)
✓ 시장수익률 평균: 5.220%

[3단계] 베타 확인
✓ Beta_5Y 평균: 0.421

[4단계] 요구수익률 계산

⚠️ 요구수익률 < Rf 인 관측치: 102개 → Rf로 대체
✓ 완료 - 평균: 3.989%, 범위: 2.648% ~ 6.244%
  → 저장할 컬럼: ['Risk_Free_Rate', 'Market_Return', 'Market_Risk_Premium', 'Required_Return', 'Beta_1Y', 'Beta_3Y', 'Beta_5Y', 'Return_20D', 'Return_60D', 'Return_120D', 'Return_252D', 'Volatility_20D', 'Volatility_60D', 'Volatility_120D', 'Volatility_252D']
  ✓ 258050 변환 완료: 27105 rows

--------------------------------------------------------------------------------
[004835] 처리 시작
--------------------------------------------------------------------------------

주주 요구수익률 계산

[1단계] 무위험이자율

[한국은행 API] 5년 국채금리 수집: 2014-01-02 ~ 2025-11-21
✓ 수집 완료 - 2930일, 평균 2.524%
✓ 한국은행 API 데이터 연동 완료
  평균: 2.524%, 범위: 1.172% ~ 4.632%

[2단계] 시장수익률 (5년)
✓ 시장수익률 평균: 3.942%

[3단계] 베타 확인
✓ Beta_5Y 평균:

Traceback (most recent call last):
  File "C:\Users\82108\AppData\Local\Temp\ipykernel_26108\2131055907.py", line 439, in run_required_return_batch
    stock_raw = fdr.DataReader(code, start_date, end_date)
  File "C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\FinanceDataReader\data.py", line 92, in DataReader
    return YahooDailyReader(codes, start, end).read()
  File "C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\FinanceDataReader\yahoo\data.py", line 61, in read
    return _yahoo_data_reader(self.symbol, self.exchange, self.start, self.end)
  File "C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\FinanceDataReader\yahoo\data.py", line 38, in _yahoo_data_reader
    r.raise_for_status()
  File "C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\requests\models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 404 Client Er


주주 요구수익률 계산

[1단계] 무위험이자율

[한국은행 API] 5년 국채금리 수집: 2014-01-02 ~ 2025-11-21
✓ 수집 완료 - 2930일, 평균 2.524%
✓ 한국은행 API 데이터 연동 완료
  평균: 2.524%, 범위: 1.172% ~ 4.632%

[2단계] 시장수익률 (5년)
✓ 시장수익률 평균: 3.942%

[3단계] 베타 확인
✓ Beta_5Y 평균: 1.007

[4단계] 요구수익률 계산

⚠️ 요구수익률 < Rf 인 관측치: 632개 → Rf로 대체
✓ 완료 - 평균: 4.744%, 범위: 1.172% ~ 12.063%
  → 저장할 컬럼: ['Risk_Free_Rate', 'Market_Return', 'Market_Risk_Premium', 'Required_Return', 'Beta_1Y', 'Beta_3Y', 'Beta_5Y', 'Return_20D', 'Return_60D', 'Return_120D', 'Return_252D', 'Volatility_20D', 'Volatility_60D', 'Volatility_120D', 'Volatility_252D']
  ✓ 001515 변환 완료: 43785 rows

--------------------------------------------------------------------------------
[473370] 처리 시작
--------------------------------------------------------------------------------
  → 상장일이 너무 최근 → 스킵

--------------------------------------------------------------------------------
[472230] 처리 시작
--------------------------------------------------------------------------------
  → 상장일이 너무 최근 → 스킵



Traceback (most recent call last):
  File "C:\Users\82108\AppData\Local\Temp\ipykernel_26108\2131055907.py", line 439, in run_required_return_batch
    stock_raw = fdr.DataReader(code, start_date, end_date)
  File "C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\FinanceDataReader\data.py", line 92, in DataReader
    return YahooDailyReader(codes, start, end).read()
  File "C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\FinanceDataReader\yahoo\data.py", line 61, in read
    return _yahoo_data_reader(self.symbol, self.exchange, self.start, self.end)
  File "C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\FinanceDataReader\yahoo\data.py", line 38, in _yahoo_data_reader
    r.raise_for_status()
  File "C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\requests\models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 404 Client Er


주주 요구수익률 계산

[1단계] 무위험이자율

[한국은행 API] 5년 국채금리 수집: 2016-04-20 ~ 2025-11-21
✓ 수집 완료 - 2363일, 평균 2.498%
✓ 한국은행 API 데이터 연동 완료
  평균: 2.498%, 범위: 1.172% ~ 4.632%

[2단계] 시장수익률 (5년)
✓ 시장수익률 평균: 4.393%

[3단계] 베타 확인
✓ Beta_5Y 평균: 0.204

[4단계] 요구수익률 계산

⚠️ 요구수익률 < Rf 인 관측치: 398개 → Rf로 대체
✓ 완료 - 평균: 3.523%, 범위: 2.627% ~ 5.019%
  → 저장할 컬럼: ['Risk_Free_Rate', 'Market_Return', 'Market_Risk_Premium', 'Required_Return', 'Beta_1Y', 'Beta_3Y', 'Beta_5Y', 'Return_20D', 'Return_60D', 'Return_120D', 'Return_252D', 'Volatility_20D', 'Volatility_60D', 'Volatility_120D', 'Volatility_252D']
  ✓ 238500 변환 완료: 35310 rows

--------------------------------------------------------------------------------
[004985] 처리 시작
--------------------------------------------------------------------------------

주주 요구수익률 계산

[1단계] 무위험이자율

[한국은행 API] 5년 국채금리 수집: 2014-01-02 ~ 2025-11-21
✓ 수집 완료 - 2930일, 평균 2.524%
✓ 한국은행 API 데이터 연동 완료
  평균: 2.524%, 범위: 1.172% ~ 4.632%

[2단계] 시장수익률 (5년)
✓ 시장수익률 평균: 3.942%

[3단계] 베타 확인
✓ Beta_5Y 평균:

In [4]:
# run_required_return_batch(
#     batch_size=10,
#     test_mode=False,
#     tickers_override=["005930", "000660", "058470", "131290", "006910",
#                       "005380", "035420", "042700", "214150", "084370"]
# )



KOSPI 지수 데이터 수집 및 분석
✓ KOSPI 데이터 수집 완료: 2919 rows

KRX 상장사 리스트 수집
⚠️ 사용자 지정 tickers 10개만 처리합니다:
['005930', '000660', '058470', '131290', '006910', '005380', '035420', '042700', '214150', '084370']
예시 tickers: ['005930', '000660', '058470', '131290', '006910', '005380', '035420', '042700', '214150', '084370']


Processing tickers:   0%|          | 0/10 [00:00<?, ?it/s]


--------------------------------------------------------------------------------
[005930] 처리 시작
--------------------------------------------------------------------------------

주주 요구수익률 계산

[1단계] 무위험이자율

[한국은행 API] 5년 국채금리 수집: 2014-01-02 ~ 2025-11-21
✓ 수집 완료 - 2930일, 평균 2.524%
✓ 한국은행 API 데이터 연동 완료
  평균: 2.524%, 범위: 1.172% ~ 4.632%

[2단계] 시장수익률 (5년)
✓ 시장수익률 평균: 3.942%

[3단계] 베타 확인
✓ Beta_5Y 평균: 1.133

[4단계] 요구수익률 계산

⚠️ 요구수익률 < Rf 인 관측치: 632개 → Rf로 대체
✓ 완료 - 평균: 5.165%, 범위: 1.172% ~ 13.311%
  → 저장할 컬럼: ['Risk_Free_Rate', 'Market_Return', 'Market_Risk_Premium', 'Required_Return', 'Beta_1Y', 'Beta_3Y', 'Beta_5Y', 'Return_20D', 'Return_60D', 'Return_120D', 'Return_252D', 'Volatility_20D', 'Volatility_60D', 'Volatility_120D', 'Volatility_252D']
  ✓ 005930 변환 완료: 43785 rows

--------------------------------------------------------------------------------
[000660] 처리 시작
--------------------------------------------------------------------------------

주주 요구수익률 계산

[1단계] 무위험이자율

[한국은행 API] 5년